# DBSCAN and Hierarchical Clustering

K-Means is honest about one limitation: it forces every point into a cluster and assumes they're roughly spherical. That works fine as a baseline, but real data rarely cooperates.

DBSCAN finds dense regions and explicitly labels sparse points as noise, which is useful for spotting responses that don't fit any archetype cleanly. Hierarchical clustering shows us the full merge tree, which tells us how the clusters relate to each other rather than just what they are.

Together, they either confirm K-Means or reveal structure it missed.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

from inference_lens.clustering.cluster import run_dbscan, run_hierarchical
from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

import os
os.makedirs('../../reports/figures', exist_ok=True)

## Load data

In [ ]:
embeddings = np.load('../../data/processed/embeddings.npy')
df = pd.read_parquet('../../data/processed/features_with_clusters.parquet')
emb_2d = np.load('../../data/processed/umap_2d.npy')
sample_idx = np.load('../../data/processed/umap_sample_idx.npy')

print(f'Embeddings: {embeddings.shape}')
print(f'Features:   {df.shape}')

## DBSCAN

DBSCAN needs two parameters: `eps` (how close two points need to be to be considered neighbors) and `min_samples` (minimum neighbors to form a dense region). Since our embeddings are L2-normalized to the unit sphere, sensible eps values are in the 0.3 to 0.8 range.

We'll try a few combinations and see what noise rate we get.

In [ ]:
# run on a 20K sample -- DBSCAN on 340K embeddings is slow without approximate methods
dbscan_idx = np.random.RandomState(42).choice(len(embeddings), 20000, replace=False)
emb_dbscan = embeddings[dbscan_idx]

print('Trying different eps values...')
for eps in [0.3, 0.4, 0.5, 0.6]:
    labels = run_dbscan(emb_dbscan, eps=eps, min_samples=15)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct = (labels == -1).mean() * 100
    print(f'  eps={eps:.1f}  clusters: {n_clusters}  noise: {noise_pct:.1f}%')

In [ ]:
# pick the eps that gives a reasonable number of clusters without too much noise
# adjust this based on the output above
best_eps = 0.5
dbscan_labels = run_dbscan(emb_dbscan, eps=best_eps, min_samples=15)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
noise_count = (dbscan_labels == -1).sum()
print(f'Final DBSCAN: {n_clusters} clusters, {noise_count} noise points ({noise_count/len(dbscan_labels)*100:.1f}%)')

## DBSCAN in UMAP space

Noise points (label = -1) are shown in grey. Real clusters get colors.

In [ ]:
# project the DBSCAN sample to 2D
import umap
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
emb_dbscan_2d = reducer.fit_transform(emb_dbscan[:5000])  # plot first 5K for speed
dbscan_plot_labels = dbscan_labels[:5000]

unique_labels = sorted(set(dbscan_plot_labels))
palette = sns.color_palette('tab10', n_colors=max(len(unique_labels), 1))

fig, ax = plt.subplots(figsize=(10, 8))
for i, label in enumerate(unique_labels):
    mask = dbscan_plot_labels == label
    color = 'lightgrey' if label == -1 else palette[i % len(palette)]
    name = 'Noise' if label == -1 else f'Cluster {label}'
    ax.scatter(emb_dbscan_2d[mask, 0], emb_dbscan_2d[mask, 1],
               s=4, alpha=0.5, color=color, label=name)

ax.set_title(f'DBSCAN clusters (eps={best_eps}, 5K sample)')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(markerscale=3, fontsize=8, loc='best')
plt.tight_layout()
plt.savefig('../../reports/figures/09_dbscan_umap.png', dpi=150, bbox_inches='tight')
plt.show()

## Hierarchical Clustering

Ward linkage minimizes within-cluster variance at each merge step, which tends to produce compact, similarly-sized clusters. The dendrogram shows you the full tree of merges so you can see at what distance different groups join together.

In [ ]:
# hierarchical clustering is memory-intensive -- run on 3000 samples
hc_idx = np.random.RandomState(42).choice(len(embeddings), 3000, replace=False)
emb_hc = embeddings[hc_idx]

print('Computing Ward linkage matrix...')
Z = linkage(emb_hc, method='ward', metric='euclidean')
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

dendrogram(
    Z,
    ax=ax,
    truncate_mode='lastp',  # show only the last p merges to keep it readable
    p=40,
    leaf_rotation=90,
    leaf_font_size=8,
    show_contracted=True,
    color_threshold=0.7 * max(Z[:, 2]),
)

ax.set_title('Hierarchical clustering dendrogram (Ward linkage, 3K sample)')
ax.set_xlabel('Sample index or cluster size')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig('../../reports/figures/10_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# cut the tree at the same K as our optimal K-Means result
optimal_k = df['kmeans_cluster'].nunique()
hc_labels = run_hierarchical(emb_hc, n_clusters=optimal_k, linkage='ward')

print(f'Hierarchical cluster sizes at K={optimal_k}:')
unique, counts = np.unique(hc_labels, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  cluster {c}: {n} points ({n/len(hc_labels)*100:.1f}%)')

## Compare K-Means and Hierarchical on the same sample

If the two methods broadly agree on cluster structure, we can trust the archetypes we name in the next notebook. If they disagree significantly, something more interesting is happening.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# get K-Means labels for the same sample used in hierarchical
kmeans_on_hc_sample = df['kmeans_cluster'].values[hc_idx]

ari = adjusted_rand_score(kmeans_on_hc_sample, hc_labels)
nmi = normalized_mutual_info_score(kmeans_on_hc_sample, hc_labels)

print(f'Adjusted Rand Index (K-Means vs Hierarchical): {ari:.4f}')
print(f'Normalized Mutual Information:                 {nmi:.4f}')
print()
print('ARI of 1.0 = perfect agreement. ARI of 0.0 = random. Negative = worse than random.')

## Save DBSCAN noise flags

Responses that DBSCAN labeled as noise are worth tracking -- they may be the ones most likely to fool evaluators in the stress-test phase.

In [ ]:
# attach noise flag to the subset we ran DBSCAN on
noise_flags = pd.Series(False, index=df.index)
noise_flags.iloc[dbscan_idx] = (dbscan_labels == -1)
df['dbscan_noise'] = noise_flags

df.to_parquet('../../data/processed/features_with_clusters.parquet', index=False)
print(f'Updated features_with_clusters.parquet with dbscan_noise column')
print(f'Noise points flagged: {df["dbscan_noise"].sum():,}')